Phase 04 : Working Data 

In [1]:
import pandas as pd

df = pd.read_csv(
    "online_retail_split.csv",
    dtype={"transaction_id": str, "product_id": str, "customer_id": str},
    parse_dates=["event_time"],
)
print(len(df))
print(df["sales_mode"].value_counts())

397884
sales_mode
Online      278740
Periodic    119144
Name: count, dtype: int64


In [2]:
ONLINE_COLS = ["transaction_id", "customer_id", "product_id",
               "quantity", "unit_price", "event_time", "sales_mode"]

online = df[df["sales_mode"] == "Online"][ONLINE_COLS]

print(len(online), "online rows")
online.head()

278740 online rows


,transaction_id,customer_id,product_id,quantity,unit_price,event_time,sales_mode
0,536365,17850,85123A,6,2.55,2010-12-01 08:26:00,Online
1,536365,17850,71053,6,3.39,2010-12-01 08:26:00,Online
2,536365,17850,84406B,8,2.75,2010-12-01 08:26:00,Online
3,536365,17850,84029G,6,3.39,2010-12-01 08:26:00,Online
4,536365,17850,84029E,6,3.39,2010-12-01 08:26:00,Online


In [ ]:
periodic = df[df["sales_mode"] == "Periodic"].copy()
total = len(periodic)

#one line per cart: its earliest time and its number of rows
carts = periodic.groupby("transaction_id").agg(
    first_time=("event_time", "min"),
    rows=("event_time", "size"),
).reset_index()

carts = carts.sort_values(["first_time", "transaction_id"]).reset_index(drop=True)

print(total, "periodic rows,", len(carts), "carts")
print("Target rows per group:", round(total / 48))

119144 periodic rows, 5561 carts
Target rows per group: 2482


In [ ]:
#where each cart starts in the sorted list (running row total)
carts["start"] = carts["rows"].cumsum() - carts["rows"]

#group = starting position * 48 / total rows  -> 0 to 47
carts["release_hour"] = carts["start"] * 48 // total

#copy the group number to every row of the cart
periodic = periodic.merge(carts[["transaction_id", "release_hour"]], on="transaction_id")
periodic["released"] = 0

STAGING_COLS = ONLINE_COLS + ["release_hour", "released"]
periodic = periodic.sort_values(["release_hour", "event_time", "transaction_id"])[STAGING_COLS]
periodic.head()

,transaction_id,customer_id,product_id,quantity,unit_price,event_time,sales_mode,release_hour,released
0,536370,12583,22728,24,3.75,2010-12-01 08:45:00,Periodic,0,0
1,536370,12583,22727,24,3.75,2010-12-01 08:45:00,Periodic,0,0
2,536370,12583,22726,12,3.75,2010-12-01 08:45:00,Periodic,0,0
3,536370,12583,21724,12,0.85,2010-12-01 08:45:00,Periodic,0,0
4,536370,12583,21883,24,0.65,2010-12-01 08:45:00,Periodic,0,0


In [ ]:
sizes = periodic.groupby("release_hour").size()

print("Online + periodic rows:", len(online) + len(periodic), "(should be", len(df), ")")
print("Number of groups:", sizes.size)
print("Group numbers:", sizes.index.min(), "to", sizes.index.max())
print("Carts split across groups:",
      (periodic.groupby("transaction_id")["release_hour"].nunique() > 1).sum())
print("Smallest group:", sizes.min(), "rows")
print("Largest group:", sizes.max(), "rows")
print("Online columns:", list(online.columns))

Online + periodic rows: 397884 (should be 397884 )
Number of groups: 48
Group numbers: 0 to 47
Carts split across groups: 0
Smallest group: 2263 rows
Largest group: 2710 rows
Online columns: ['transaction_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'event_time', 'sales_mode']


In [6]:
online.to_csv("online_sales_stream.csv", index=False)
periodic.to_csv("periodic_sales_data.csv", index=False)
print("Saved:", len(online), "online rows,", len(periodic), "periodic rows")

Saved: 278740 online rows, 119144 periodic rows
